# Exp 005 Native Priors Texture Postproc

## Goal

Measure how much `site/hour/site-hour priors` and `texture-aware smoothing` improve the native `exp_004` soundscape model without any new training.

## Question

Is the remaining gap after `exp_004` mostly a modeling problem or a soundscape calibration problem?

## What This Experiment Tests

- raw `exp_004` validation predictions
- event priors only
- texture priors only
- event + texture priors
- event + texture priors + texture smoothing

## Why This Matters

If these cheap soundscape-aware postprocessing steps help the native branch meaningfully, then the next best native hybrid is likely `exp_004 + priors`, not a much heavier pseudo-label or second-stage stacker branch.


## Expected Readout

The notebook saves:

- `result_snapshot.json`
- `ablation_results.csv`
- `classwise_auc_comparison.csv`
- `valid_predictions.parquet`

All outputs are written to `experiments/outputs/exp_005_native_priors_texture_postproc/`.


In [ ]:
from __future__ import annotations

import json
import sys
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None
for candidate in PROJECT_ROOT_CANDIDATES:
    if (candidate / 'data' / 'birdclef-2026').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not resolve PROJECT_ROOT with data/birdclef-2026')

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from birdclef2026.reference_eval import (  # noqa: E402
    ReferenceModelConfig,
    build_reference_model_components,
    score_macro_auc,
)


@dataclass
class EvalConfig:
    seed: int = 42
    sample_rate: int = 32_000
    clip_duration: float = 5.0
    batch_size: int = 12
    num_workers: int = 0
    fold: int = 0
    n_folds: int = 5
    use_amp: bool = True
    lambda_event: float = 0.4
    lambda_texture: float = 1.0
    smooth_texture: float = 0.35

    @property
    def clip_samples(self) -> int:
        return int(self.sample_rate * self.clip_duration)


CFG = EvalConfig()
DATA_DIR = PROJECT_ROOT / 'data' / 'birdclef-2026'
TRAIN_SOUNDSCAPE_DIR = DATA_DIR / 'train_soundscapes'
EXP004_DIR = PROJECT_ROOT / 'experiments' / 'outputs' / 'exp_004_soundscape_finetuning'
OUTPUT_DIR = PROJECT_ROOT / 'experiments' / 'outputs' / 'exp_005_native_priors_texture_postproc'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
amp_enabled = CFG.use_amp and device.type == 'cuda'

np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
torch.set_float32_matmul_precision('high')

{
    'project_root': str(PROJECT_ROOT),
    'device': str(device),
    'amp_enabled': amp_enabled,
    'output_dir': str(OUTPUT_DIR),
    **asdict(CFG),
}


In [ ]:
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
soundscape_labels = pd.read_csv(DATA_DIR / 'train_soundscapes_labels.csv')
taxonomy = pd.read_csv(DATA_DIR / 'taxonomy.csv')

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
label_to_idx = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}
TEXTURE_TAXA = {'Amphibia', 'Insecta'}


def parse_soundscape_labels(value) -> list[str]:
    if pd.isna(value):
        return []
    return [token.strip() for token in str(value).split(';') if token.strip()]


def union_labels(series: pd.Series) -> list[str]:
    return sorted(set(label for item in series for label in parse_soundscape_labels(item)))


def parse_soundscape_filename(name: str) -> dict:
    stem = Path(name).stem
    parts = stem.split('_')
    site = parts[3] if len(parts) >= 4 else None
    time_token = parts[-1] if parts else '000000'
    hour_utc = int(time_token[:2]) if time_token.isdigit() and len(time_token) >= 2 else -1
    return {'site': site, 'hour_utc': hour_utc}


def encode_multi_hot(labels: list[str]) -> np.ndarray:
    target = np.zeros(len(PRIMARY_LABELS), dtype=np.float32)
    for label in labels:
        idx = label_to_idx.get(label)
        if idx is not None:
            target[idx] = 1.0
    return target


sc_clean = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc_clean['start_sec'] = pd.to_timedelta(sc_clean['start']).dt.total_seconds().astype(int)
sc_clean['end_sec'] = pd.to_timedelta(sc_clean['end']).dt.total_seconds().astype(int)
sc_clean['row_id'] = sc_clean['filename'].str.replace('.ogg', '', regex=False) + '_' + sc_clean['end_sec'].astype(str)
sc_meta = sc_clean['filename'].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, sc_meta], axis=1)
sc_clean['file_path'] = sc_clean['filename'].apply(lambda name: TRAIN_SOUNDSCAPE_DIR / name)
sc_clean = sc_clean[sc_clean['file_path'].map(Path.exists)].reset_index(drop=True)

windows_per_file = sc_clean.groupby('filename').size()
full_files = sorted(windows_per_file[windows_per_file == 12].index.tolist())
full_df = sc_clean[sc_clean['filename'].isin(full_files)].copy()

n_group_splits = min(CFG.n_folds, max(2, len(full_files)))
gkf = GroupKFold(n_splits=n_group_splits)
full_splits = list(gkf.split(full_df, groups=full_df['filename']))
train_full_idx, valid_full_idx = full_splits[CFG.fold % len(full_splits)]
valid_files = set(full_df.iloc[valid_full_idx]['filename'].tolist())

train_df = sc_clean[~sc_clean['filename'].isin(valid_files)].reset_index(drop=True)
valid_df = (
    full_df[full_df['filename'].isin(valid_files)]
    .sort_values(['filename', 'end_sec'])
    .reset_index(drop=True)
)

Y_TRAIN = np.stack([encode_multi_hot(labels) for labels in train_df['label_list']], axis=0)
Y_VALID = np.stack([encode_multi_hot(labels) for labels in valid_df['label_list']], axis=0)

texture_labels = taxonomy.loc[taxonomy['class_name'].isin(TEXTURE_TAXA), 'primary_label'].tolist()
idx_texture = np.array([label_to_idx[label] for label in texture_labels if label in label_to_idx], dtype=np.int32)
idx_event = np.array([idx for idx, label in enumerate(PRIMARY_LABELS) if label not in set(texture_labels)], dtype=np.int32)

{
    'all_labeled_segments': int(len(sc_clean)),
    'fully_labeled_files': int(len(full_files)),
    'train_segments': int(len(train_df)),
    'valid_segments': int(len(valid_df)),
    'valid_files': int(len(valid_files)),
    'texture_classes': int(len(idx_texture)),
    'event_classes': int(len(idx_event)),
    'scored_classes_if_raw': int((Y_VALID.sum(axis=0) > 0).sum()),
}


In [ ]:
class SoundscapeEvalDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, clip_samples: int, sample_rate: int):
        self.frame = frame.reset_index(drop=True)
        self.clip_samples = clip_samples
        self.sample_rate = sample_rate

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        audio, sr = sf.read(row['file_path'], dtype='float32', always_2d=False)
        if audio.ndim == 2:
            audio = audio.mean(axis=1)
        if sr != self.sample_rate:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=self.sample_rate)
        audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)

        start = int(row['start_sec'] * self.sample_rate)
        end = start + self.clip_samples
        clip = audio[start:end]
        if len(clip) < self.clip_samples:
            clip = np.pad(clip, (0, self.clip_samples - len(clip)))

        return {
            'audio': torch.from_numpy(clip.astype(np.float32)),
            'row_id': row['row_id'],
            'filename': row['filename'],
            'site': row['site'],
            'hour_utc': int(row['hour_utc']),
            'end_sec': int(row['end_sec']),
        }


model_cfg = ReferenceModelConfig(num_classes=len(PRIMARY_LABELS))


class WaveformSEDClassifier(nn.Module):
    def __init__(self, model_cfg: ReferenceModelConfig):
        super().__init__()
        SEDModelCls, MelTransformCls = build_reference_model_components(model_cfg)
        self.mel = MelTransformCls()
        self.model = SEDModelCls()

    def forward(self, waveforms: torch.Tensor) -> dict[str, torch.Tensor]:
        mel = self.mel(waveforms)
        return self.model(mel)


def load_model_checkpoint(path: Path, model) -> dict:
    checkpoint = torch.load(path, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    return checkpoint


def sigmoid(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32, copy=False)
    positive = x >= 0
    out = np.empty_like(x, dtype=np.float32)
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    out[~positive] = exp_x / (1.0 + exp_x)
    return out


def summarize_macro_auc(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    metrics = score_macro_auc(y_true, y_pred, PRIMARY_LABELS)
    return {
        'macro_auc': metrics['macro_auc'],
        'scored_classes': metrics['scored_classes'],
        'skipped_no_positive': metrics['skipped_no_positive'],
        'skipped_no_negative': metrics['skipped_no_negative'],
    }


In [ ]:
checkpoint_path = EXP004_DIR / 'best_model.pt'
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Missing exp_004 checkpoint: {checkpoint_path}')

valid_dataset = SoundscapeEvalDataset(valid_df, clip_samples=CFG.clip_samples, sample_rate=CFG.sample_rate)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=device.type == 'cuda',
    drop_last=False,
)

model = WaveformSEDClassifier(model_cfg).to(device)
checkpoint_meta = load_model_checkpoint(checkpoint_path, model)
model.eval()

all_logits = []
all_row_ids = []
all_filenames = []
all_sites = []
all_hours = []
all_end_sec = []

with torch.no_grad():
    for batch in tqdm(valid_loader, desc='Infer exp_004 valid fold', leave=False):
        audio = batch['audio'].to(device, non_blocking=True)
        with (torch.amp.autocast(device_type=device.type, enabled=amp_enabled) if amp_enabled else nullcontext()):
            output = model(audio)
        logits = output['clipwise_logit'].detach().cpu().numpy().astype(np.float32)
        all_logits.append(logits)
        all_row_ids.extend(list(batch['row_id']))
        all_filenames.extend(list(batch['filename']))
        all_sites.extend(list(batch['site']))
        all_hours.extend([int(x) for x in batch['hour_utc']])
        all_end_sec.extend([int(x) for x in batch['end_sec']])

valid_logits = np.concatenate(all_logits, axis=0)
valid_probs = sigmoid(valid_logits)

pred_meta = pd.DataFrame({
    'row_id': all_row_ids,
    'filename': all_filenames,
    'site': all_sites,
    'hour_utc': all_hours,
    'end_sec': all_end_sec,
}).sort_values(['filename', 'end_sec']).reset_index(drop=True)

sort_index = pred_meta.index.to_numpy()
valid_logits = valid_logits[sort_index]
valid_probs = valid_probs[sort_index]
Y_VALID_SORTED = Y_VALID[sort_index]

try:
    pred_meta.to_parquet(OUTPUT_DIR / 'valid_prediction_meta.parquet', index=False)
except ImportError:
    pred_meta.to_csv(OUTPUT_DIR / 'valid_prediction_meta.csv', index=False)

np.savez_compressed(
    OUTPUT_DIR / 'valid_raw_predictions.npz',
    valid_logits=valid_logits,
    valid_probs=valid_probs,
    y_valid=Y_VALID_SORTED,
)

{
    'checkpoint_epoch': int(checkpoint_meta.get('epoch', -1)),
    'checkpoint_macro_auc': checkpoint_meta.get('metrics', {}).get('macro_auc'),
    'valid_rows': int(len(pred_meta)),
    'raw_macro_auc': summarize_macro_auc(Y_VALID_SORTED, valid_probs)['macro_auc'],
}


In [ ]:
def fit_prior_tables(prior_df: pd.DataFrame, y_prior: np.ndarray) -> dict:
    prior_df = prior_df.reset_index(drop=True)
    global_p = y_prior.mean(axis=0).astype(np.float32)

    site_keys = sorted(prior_df['site'].dropna().astype(str).unique().tolist())
    site_to_i = {key: idx for idx, key in enumerate(site_keys)}
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    site_p = np.zeros((len(site_keys), y_prior.shape[1]), dtype=np.float32)
    for site in site_keys:
        idx = site_to_i[site]
        mask = prior_df['site'].astype(str).values == site
        site_n[idx] = mask.sum()
        site_p[idx] = y_prior[mask].mean(axis=0)

    hour_keys = sorted(prior_df['hour_utc'].dropna().astype(int).unique().tolist())
    hour_to_i = {hour: idx for idx, hour in enumerate(hour_keys)}
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    hour_p = np.zeros((len(hour_keys), y_prior.shape[1]), dtype=np.float32)
    for hour in hour_keys:
        idx = hour_to_i[hour]
        mask = prior_df['hour_utc'].astype(int).values == hour
        hour_n[idx] = mask.sum()
        hour_p[idx] = y_prior[mask].mean(axis=0)

    sh_to_i = {}
    sh_n_list = []
    sh_p_list = []
    for (site, hour), group_idx in prior_df.groupby(['site', 'hour_utc']).groups.items():
        sh_to_i[(str(site), int(hour))] = len(sh_n_list)
        group_idx = np.array(list(group_idx))
        sh_n_list.append(len(group_idx))
        sh_p_list.append(y_prior[group_idx].mean(axis=0))

    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = np.stack(sh_p_list).astype(np.float32) if len(sh_p_list) else np.zeros((0, y_prior.shape[1]), dtype=np.float32)

    return {
        'global_p': global_p,
        'site_to_i': site_to_i,
        'site_n': site_n,
        'site_p': site_p,
        'hour_to_i': hour_to_i,
        'hour_n': hour_n,
        'hour_p': hour_p,
        'sh_to_i': sh_to_i,
        'sh_n': sh_n,
        'sh_p': sh_p,
    }


def prior_logits_from_tables(sites, hours, tables, eps=1e-4):
    n_rows = len(sites)
    p = np.repeat(tables['global_p'][None, :], n_rows, axis=0).astype(np.float32, copy=True)

    site_idx = np.fromiter((tables['site_to_i'].get(str(site), -1) for site in sites), dtype=np.int32, count=n_rows)
    hour_idx = np.fromiter((tables['hour_to_i'].get(int(hour), -1) if int(hour) >= 0 else -1 for hour in hours), dtype=np.int32, count=n_rows)
    sh_idx = np.fromiter(
        (tables['sh_to_i'].get((str(site), int(hour)), -1) if int(hour) >= 0 else -1 for site, hour in zip(sites, hours)),
        dtype=np.int32,
        count=n_rows,
    )

    valid = hour_idx >= 0
    if valid.any():
        nh = tables['hour_n'][hour_idx[valid]][:, None]
        wh = nh / (nh + 8.0)
        p[valid] = wh * tables['hour_p'][hour_idx[valid]] + (1.0 - wh) * p[valid]

    valid = site_idx >= 0
    if valid.any():
        ns = tables['site_n'][site_idx[valid]][:, None]
        ws = ns / (ns + 8.0)
        p[valid] = ws * tables['site_p'][site_idx[valid]] + (1.0 - ws) * p[valid]

    valid = sh_idx >= 0
    if valid.any():
        nsh = tables['sh_n'][sh_idx[valid]][:, None]
        wsh = nsh / (nsh + 4.0)
        p[valid] = wsh * tables['sh_p'][sh_idx[valid]] + (1.0 - wsh) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32, copy=False)


def smooth_cols_fixed12(scores: np.ndarray, cols: np.ndarray, alpha: float = 0.35) -> np.ndarray:
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    smoothed = scores.copy()
    assert len(smoothed) % 12 == 0
    view = smoothed.reshape(-1, 12, smoothed.shape[1])
    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
    return smoothed


def fuse_native_logits(logits: np.ndarray, prior_logits: np.ndarray, lambda_event: float, lambda_texture: float, smooth_texture: float = 0.0) -> np.ndarray:
    fused = logits.copy()
    if len(idx_event):
        fused[:, idx_event] += lambda_event * prior_logits[:, idx_event]
    if len(idx_texture):
        fused[:, idx_texture] += lambda_texture * prior_logits[:, idx_texture]
    if smooth_texture > 0:
        fused = smooth_cols_fixed12(fused, idx_texture, alpha=smooth_texture)
    return fused


def classwise_auc_table(y_true: np.ndarray, y_score: np.ndarray, labels: list[str]) -> pd.DataFrame:
    rows = []
    for idx, label in enumerate(labels):
        positives = int(y_true[:, idx].sum())
        negatives = int(len(y_true) - positives)
        if positives == 0 or negatives == 0:
            auc = np.nan
        else:
            auc = float(score_macro_auc(y_true[:, [idx]], y_score[:, [idx]], [label])['macro_auc'])
        rows.append({
            'primary_label': label,
            'positives': positives,
            'negatives': negatives,
            'auc': auc,
        })
    return pd.DataFrame(rows)


In [ ]:
prior_tables = fit_prior_tables(train_df[['site', 'hour_utc']], Y_TRAIN)
prior_logits = prior_logits_from_tables(
    sites=pred_meta['site'].to_numpy(),
    hours=pred_meta['hour_utc'].to_numpy(),
    tables=prior_tables,
)

ablations = []
variant_scores = {}

raw_probs = valid_probs.copy()
variant_scores['raw'] = raw_probs
ablations.append({'variant': 'raw', **summarize_macro_auc(Y_VALID_SORTED, raw_probs)})

variant_settings = [
    ('event_priors_only', CFG.lambda_event, 0.0, 0.0),
    ('texture_priors_only', 0.0, CFG.lambda_texture, 0.0),
    ('event_texture_priors', CFG.lambda_event, CFG.lambda_texture, 0.0),
    ('event_texture_priors_smooth', CFG.lambda_event, CFG.lambda_texture, CFG.smooth_texture),
]

for name, lambda_event, lambda_texture, smooth_texture in variant_settings:
    fused_logits = fuse_native_logits(
        logits=valid_logits,
        prior_logits=prior_logits,
        lambda_event=lambda_event,
        lambda_texture=lambda_texture,
        smooth_texture=smooth_texture,
    )
    fused_probs = sigmoid(fused_logits)
    variant_scores[name] = fused_probs
    ablations.append({'variant': name, **summarize_macro_auc(Y_VALID_SORTED, fused_probs)})

ablation_df = pd.DataFrame(ablations).sort_values('macro_auc', ascending=False).reset_index(drop=True)
ablation_df.to_csv(OUTPUT_DIR / 'ablation_results.csv', index=False)

best_variant = ablation_df.iloc[0]['variant']
best_scores = variant_scores[best_variant]

classwise_raw = classwise_auc_table(Y_VALID_SORTED, raw_probs, PRIMARY_LABELS).rename(columns={'auc': 'raw_auc'})
classwise_best = classwise_auc_table(Y_VALID_SORTED, best_scores, PRIMARY_LABELS).rename(columns={'auc': 'best_auc'})
classwise_compare = classwise_raw.merge(classwise_best[['primary_label', 'best_auc']], on='primary_label', how='left')
classwise_compare['delta'] = classwise_compare['best_auc'] - classwise_compare['raw_auc']
classwise_compare.to_csv(OUTPUT_DIR / 'classwise_auc_comparison.csv', index=False)

pred_export = pred_meta.copy()
pred_export['best_variant'] = best_variant
for idx, label in enumerate(PRIMARY_LABELS):
    pred_export[f'raw__{label}'] = raw_probs[:, idx].astype(np.float32)
    pred_export[f'{best_variant}__{label}'] = best_scores[:, idx].astype(np.float32)
try:
    pred_export.to_parquet(OUTPUT_DIR / 'valid_predictions.parquet', index=False)
except ImportError:
    pred_export.to_csv(OUTPUT_DIR / 'valid_predictions.csv', index=False)

result_snapshot = {
    'experiment_id': 'exp_005',
    'experiment_name': 'native_priors_texture_postproc',
    'source_checkpoint': str(checkpoint_path),
    'fold': int(CFG.fold),
    'raw_macro_auc': float(ablation_df.loc[ablation_df['variant'] == 'raw', 'macro_auc'].iloc[0]),
    'best_variant': str(best_variant),
    'best_macro_auc': float(ablation_df.iloc[0]['macro_auc']),
    'delta_vs_raw': float(ablation_df.iloc[0]['macro_auc'] - ablation_df.loc[ablation_df['variant'] == 'raw', 'macro_auc'].iloc[0]),
    'lambda_event': float(CFG.lambda_event),
    'lambda_texture': float(CFG.lambda_texture),
    'smooth_texture': float(CFG.smooth_texture),
    'scored_classes_best_variant': int(ablation_df.iloc[0]['scored_classes']),
}
(OUTPUT_DIR / 'result_snapshot.json').write_text(json.dumps(result_snapshot, indent=2))

print(ablation_df)
print('\nSnapshot:')
print(json.dumps(result_snapshot, indent=2))


In [ ]:
snapshot = json.loads((OUTPUT_DIR / 'result_snapshot.json').read_text())
print('Snapshot:')
print(json.dumps(snapshot, indent=2))

print('\nAblations:')
display(pd.read_csv(OUTPUT_DIR / 'ablation_results.csv'))

compare_df = pd.read_csv(OUTPUT_DIR / 'classwise_auc_comparison.csv')
print('\nTop gains:')
display(compare_df.sort_values('delta', ascending=False).head(20))
print('\nTop regressions:')
display(compare_df.sort_values('delta', ascending=True).head(20))


## Reading The Result

The most important question is not whether the absolute number beats `exp_003` immediately.

The most important question is whether the native branch gets a **meaningful uplift over raw `exp_004`** from the same kinds of priors and texture logic that helped the Perch branch.

If the uplift is real, the next native hybrid should likely be:

1. stronger soundscape validation coverage
2. `exp_004` + priors as the default native inference layer
3. only after that, decide whether a heavier stacker or pseudo-label branch is worth the added complexity
